In [2]:
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_lo

In [45]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

In [6]:
import torch.nn as nn 
import torch.optim as optim 

## XOR classifier problem

In [26]:
X = torch.tensor([[0,0],
                 [0,1],
                 [1,0],
                 [1,1]],dtype=torch.float32)

y = torch.tensor([[0],
                  [1],
                  [1],
                  [0]],dtype=torch.float32)

print(f"x: {X}")
print(f"y: {y}")
print(f"y shape: {y.shape}")

x: tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
y: tensor([[0.],
        [1.],
        [1.],
        [0.]])
y shape: torch.Size([4, 1])


In [27]:
class XORModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)   # 2 features and 4 hidden layers
        self.fc2 = nn.Linear(4, 1)   # 4 hidden layers and 1 output layer

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x

In [28]:
model = XORModel()
model(X)

tensor([[0.4350],
        [0.4333],
        [0.4161],
        [0.4093]], grad_fn=<SigmoidBackward0>)

In [29]:
model(X).shape

torch.Size([4, 1])

In [30]:
model.fc1.bias

Parameter containing:
tensor([ 0.6149, -0.5308, -0.4099,  0.3726], requires_grad=True)

In [31]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

In [39]:
for epoch in range(3000):
    #1.forward pass
    prediction = model(X)

    #2. loss calculation
    loss = criterion(prediction, y)

    #3. zero gradients
    optimizer.zero_grad()

    #4. backward propagation
    loss.backward()

    #5. update weights
    optimizer.step()

    if(epoch+1)%500==0:
        print(f"Epoch: {epoch+1} | Loss: {loss:.10f}")

Epoch: 500 | Loss: 0.0000000458
Epoch: 1000 | Loss: 0.0000000356
Epoch: 1500 | Loss: 0.0000000277
Epoch: 2000 | Loss: 0.0000000215
Epoch: 2500 | Loss: 0.0000000168
Epoch: 3000 | Loss: 0.0000000130


In [40]:
!pip3 install torchinfo

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12 -m pip install --upgrade pip


In [41]:
from torchinfo import summary

summary(XORModel())

Layer (type:depth-idx)                   Param #
XORModel                                 --
├─Linear: 1-1                            12
├─Linear: 1-2                            5
Total params: 17
Trainable params: 17
Non-trainable params: 0

## Regression : Fit a Sine wave

In [42]:
X = torch.linspace(-3.14, 3.14, 1000).unsqueeze(1)
y = torch.sin(X) + 0.1 * torch.randn_like(X)

In [50]:
y.shape

torch.Size([1000, 1])

In [58]:
class SineModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [59]:
model = SineModel()

In [ ]:
model(X)

torch.Size([1000, 1])

In [68]:
model.fc1.bias.shape

torch.Size([64])

In [70]:
model.parameters

<bound method Module.parameters of SineModel(
  (fc1): Linear(in_features=1, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)>

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=200, min_lr=1e-5, factor=0.5, verbose=True)

/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [90]:
for epoch in range(5000):
    #1. forward pass
    prediction = model(X)

    #2. loss calculation 
    loss = criterion(prediction, y)

    #3. gradient zero
    optimizer.zero_grad()
    
    #4. backward pass
    loss.backward()

    #5. upgrade weights
    optimizer.step()

    # #6. learning rate scheduling
    # scheduler.step(loss)

    if((epoch+1)%500==0):
        print(f"Epoch: {epoch+1} | Loss: {loss:.10f} ")
    


Epoch: 500 | Loss: 0.0085798511 
Epoch: 1000 | Loss: 0.0085786665 
Epoch: 1500 | Loss: 0.0085775033 
Epoch: 2000 | Loss: 0.0085763810 
Epoch: 2500 | Loss: 0.0085749105 
Epoch: 3000 | Loss: 0.0085738311 
Epoch: 3500 | Loss: 0.0085727582 
Epoch: 4000 | Loss: 0.0085716862 
Epoch: 4500 | Loss: 0.0085706310 
Epoch: 5000 | Loss: 0.0085696019 


In [ ]:
summary(SineModel())

Layer (type:depth-idx)                   Param #
SineModel                                --
├─Linear: 1-1                            128
├─Linear: 1-2                            4,160
├─Linear: 1-3                            65
Total params: 4,353
Trainable params: 4,353
Non-trainable params: 0